# 🛡️ NetSentinel — Detector 2: DGA / DNS Tunneling (CNN-BiLSTM)

**Model**: 1D-CNN → BiLSTM (industry standard for DGA detection) 
**Dataset**: DGArchive (malicious) + Tranco Top-1M (benign) 
**Task**: 3-class: Benign vs DGA vs DNS Tunnel 
**Export**: ONNX 
**Training Time**: ~1-2 hours on T4 GPU 

**Why CNN-BiLSTM?** This is the architecture used by Cisco Umbrella and Endgame (now Elastic). 
The CNN captures local character n-gram patterns, the BiLSTM captures sequential dependencies 
across the entire domain string. It beats standalone CNNs on wordlist-based DGAs (Suppobox, Matsnu).

---

In [ ]:
!pip install -q torch torchvision onnx onnxruntime scikit-learn pandas matplotlib seaborn tqdm

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import os, json, time, string, math, random
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')

OUTPUT_DIR = '/kaggle/working/output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 1. Load & Prepare Domain Data

**Data sources:**
- **Benign**: Tranco Top-1M list (`tranco-list.eu`)
- **DGA**: DGArchive or Kaggle DGA datasets
- **DNS Tunnel**: Generate synthetic or use iodine/dns2tcp samples

For Kaggle, search for:
- `kaggle.com/datasets` → search "DGA domains" or "domain generation algorithm"
- Popular ones: `dga-domains-dataset`, `dgta-benchmark`

In [ ]:
# ============================================================
# CONFIGURE PATHS
# ============================================================

# Option A: Kaggle
# BENIGN_FILE = '/kaggle/input/tranco-top1m/top-1m.csv'
# DGA_FILE = '/kaggle/input/dga-domains/dga_domains.csv'

# Option B: Local
BENIGN_FILE = r'D:\1_sih26\dataset\tranco_top1m.csv'  # Download from tranco-list.eu
DGA_FILE = r'D:\1_sih26\dataset\dga_domains.csv'       # Download from DGArchive or Kaggle

# If you don't have files yet, we'll generate synthetic data to get started
USE_SYNTHETIC = False


In [ ]:
# ============================================================
# Data Loading (Kaggle)
# ============================================================
import glob
import os
import pandas as pd
import string
import random

def generate_tunnel_domains(n=20000):
    """Generate DNS tunneling-style domains (long, encoded subdomains)."""
    base_domains = ['tunnel-cdn.com', 'data-sync.net', 'update-svc.org', 'cloud-relay.io']
    domains = []
    for _ in range(n):
        n_labels = random.randint(2, 5)
        labels = []
        for _ in range(n_labels):
            label_len = random.randint(10, 60)
            label = ''.join(random.choices(string.ascii_lowercase + string.digits, k=label_len))
            labels.append(label)
        domain = '.'.join(labels) + '.' + random.choice(base_domains)
        domains.append(domain)
    return domains

print('Loading real domain data from Kaggle inputs...')
gtk_files = glob.glob('/kaggle/input/datasets/gtkcyber/dga-dataset/**/*.csv', recursive=True)
umudga_files = glob.glob('/kaggle/input/datasets/saurabhshahane/domain-generation/**/*.csv', recursive=True)
all_csvs = gtk_files + umudga_files

benign_list = []
dga_list = []

for f in all_csvs:
    try:
        print(f'Reading {f}...')
        df_tmp = pd.read_csv(f, usecols=lambda c: 'domain' in c.lower() or 'isdga' in c.lower() or 'class' in c.lower() or 'label' in c.lower())
        
        domain_col = next((c for c in df_tmp.columns if 'domain' in c.lower()), None)
        label_col = next((c for c in df_tmp.columns if 'isdga' in c.lower() or 'class' in c.lower() or 'label' in c.lower()), None)
        
        if domain_col and label_col:
            # Handle string classes like 'dga' or 'legit'
            is_dga = df_tmp[label_col].apply(lambda x: 1 if str(x).lower() in ['1', 'dga', 'true', 'malicious'] else 0)
            
            benign_list.extend(df_tmp[is_dga == 0][domain_col].dropna().astype(str).tolist())
            dga_list.extend(df_tmp[is_dga == 1][domain_col].dropna().astype(str).tolist())
            
        elif domain_col:
            # If no label col, assume it's DGA based on filename or just skip
            if 'legit' in f.lower() or 'benign' in f.lower():
                benign_list.extend(df_tmp[domain_col].dropna().astype(str).tolist())
            else:
                dga_list.extend(df_tmp[domain_col].dropna().astype(str).tolist())
    except Exception as e:
        print(f'Error reading {f}: {e}')

# Limit size so Kaggle RAM doesn't crash
MAX_SAMPLES = 250000
benign_list = list(set(benign_list))[:MAX_SAMPLES]
dga_list = list(set(dga_list))[:MAX_SAMPLES]
tunnel_list = generate_tunnel_domains(int(MAX_SAMPLES * 0.2))  # 20% of max samples for tunnel

domains = benign_list + dga_list + tunnel_list
labels = [0]*len(benign_list) + [1]*len(dga_list) + [2]*len(tunnel_list)

print(f'\nTotal domains: {len(domains):,}')
print(f'  Benign: {len(benign_list):,}')
print(f'  DGA:    {len(dga_list):,}')
print(f'  Tunnel: {len(tunnel_list):,}')


## 2. Character-Level Encoding

In [ ]:
# ============================================================
# Character vocabulary: a-z, 0-9, -, . + padding + unknown
# ============================================================

CHARS = list('abcdefghijklmnopqrstuvwxyz0123456789-.')
CHAR2IDX = {c: i+2 for i, c in enumerate(CHARS)}  # 0=pad, 1=unknown
CHAR2IDX['<PAD>'] = 0
CHAR2IDX['<UNK>'] = 1
VOCAB_SIZE = len(CHAR2IDX)
MAX_LEN = 253  # Max DNS domain length

# For practical purposes, cap at 128 chars (99%+ of domains are shorter)
MAX_LEN = 128

def encode_domain(domain, max_len=MAX_LEN):
    """Convert domain string to integer sequence."""
    domain = domain.lower().strip()
    encoded = [CHAR2IDX.get(c, 1) for c in domain[:max_len]]
    # Pad to max_len
    encoded += [0] * (max_len - len(encoded))
    return encoded

# Test
test_domains = ['google.com', 'xk4jf9a2m.biz', 'aGVsbG8.d29ybGQ.tunnel-cdn.com']
for d in test_domains:
    enc = encode_domain(d)
    print(f'{d:40s} → len={len(d):3d}, encoded[:10]={enc[:10]}')

print(f'\nVocab size: {VOCAB_SIZE}')
print(f'Max sequence length: {MAX_LEN}')

# Save vocab for inference
with open(os.path.join(OUTPUT_DIR, 'char_vocab.json'), 'w') as f:
    json.dump(CHAR2IDX, f)

In [ ]:
# ============================================================
# Create PyTorch Dataset
# ============================================================

class DomainDataset(Dataset):
    def __init__(self, domains, labels):
        self.domains = domains
        self.labels = labels
    
    def __len__(self):
        return len(self.domains)
    
    def __getitem__(self, idx):
        encoded = encode_domain(self.domains[idx])
        return (
            torch.tensor(encoded, dtype=torch.long),
            torch.tensor(self.labels[idx], dtype=torch.long)
        )

# Split data
X_train_d, X_test_d, y_train, y_test = train_test_split(
    domains, labels, test_size=0.2, random_state=SEED, stratify=labels
)

train_ds = DomainDataset(X_train_d, y_train)
test_ds = DomainDataset(X_test_d, y_test)

BATCH_SIZE = 512
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {len(train_ds):,} samples, {len(train_loader)} batches')
print(f'Test:  {len(test_ds):,} samples, {len(test_loader)} batches')

## 3. Model Architecture: CNN-BiLSTM

In [ ]:
# ============================================================
# CNN-BiLSTM: Industry-standard DGA Detection Architecture
# 
# References:
# - Woodbridge et al. "Detecting DGA with LSTMs" (2016) - Endgame
# - Yu et al. "Character-level CNN for DGA" (2018) - Cisco Umbrella
# - Combined CNN+LSTM approach from multiple follow-up papers
# ============================================================

class DGADetector(nn.Module):
    def __init__(
        self,
        vocab_size=VOCAB_SIZE,
        embed_dim=64,
        cnn_filters=128,
        cnn_kernel_sizes=[3, 4, 5],
        lstm_hidden=64,
        lstm_layers=2,
        n_classes=3,
        dropout=0.3
    ):
        super().__init__()
        
        # Character embedding
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        
        # Multi-kernel CNN (captures different n-gram sizes)
        self.convs = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(embed_dim, cnn_filters, kernel_size=k, padding='same'),
                nn.BatchNorm1d(cnn_filters),
                nn.ReLU(),
                nn.Dropout(dropout)
            )
            for k in cnn_kernel_sizes
        ])
        
        # BiLSTM on CNN output
        cnn_out_dim = cnn_filters * len(cnn_kernel_sizes)
        self.lstm = nn.LSTM(
            input_size=cnn_out_dim,
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if lstm_layers > 1 else 0
        )
        
        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(lstm_hidden * 2, 128),  # *2 for bidirectional
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(64, n_classes)
        )
        
    def forward(self, x):
        # x: [batch, seq_len] (character indices)
        embedded = self.embedding(x)  # [batch, seq_len, embed_dim]
        embedded = embedded.transpose(1, 2)  # [batch, embed_dim, seq_len] for Conv1d
        
        # Apply each CNN kernel and concatenate
        conv_outs = [conv(embedded) for conv in self.convs]  # each: [batch, filters, seq_len]
        cnn_out = torch.cat(conv_outs, dim=1)  # [batch, filters*n_kernels, seq_len]
        cnn_out = cnn_out.transpose(1, 2)  # [batch, seq_len, filters*n_kernels] for LSTM
        
        # BiLSTM
        lstm_out, (h_n, _) = self.lstm(cnn_out)
        # Use last hidden state from both directions
        h_forward = h_n[-2]  # Last layer forward
        h_backward = h_n[-1]  # Last layer backward
        h_combined = torch.cat([h_forward, h_backward], dim=1)  # [batch, hidden*2]
        
        # Classify
        logits = self.classifier(h_combined)  # [batch, n_classes]
        return logits

model = DGADetector().to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTotal params: {total_params:,}')
print(f'Trainable:    {trainable_params:,}')


## 4. Training

In [ ]:
# ============================================================
# Training configuration
# ============================================================

EPOCHS = 20
LR = 0.001

# Class weights for imbalanced data
class_counts = Counter(y_train)
total = sum(class_counts.values())
class_weights = torch.tensor(
    [total / (len(class_counts) * class_counts[i]) for i in range(3)],
    dtype=torch.float32
).to(device)
print(f'Class weights: {class_weights}')

criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

In [ ]:
# ============================================================
# Training loop
# ============================================================

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'val_f1': []}
best_f1 = 0.0

print('Starting training...')
start_time = time.time()

for epoch in range(EPOCHS):
    # ---- Train ----
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0
    
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS}', leave=False)
    for batch_x, batch_y in pbar:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        logits = model(batch_x)
        loss = criterion(logits, batch_y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        train_loss += loss.item() * batch_x.size(0)
        train_correct += (logits.argmax(1) == batch_y).sum().item()
        train_total += batch_x.size(0)
        
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{train_correct/train_total:.4f}'})
    
    # ---- Validate ----
    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            
            val_loss += loss.item() * batch_x.size(0)
            preds = logits.argmax(1)
            val_correct += (preds == batch_y).sum().item()
            val_total += batch_x.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(batch_y.cpu().numpy())
    
    scheduler.step()
    
    # Metrics
    train_loss_avg = train_loss / train_total
    val_loss_avg = val_loss / val_total
    train_acc = train_correct / train_total
    val_acc = val_correct / val_total
    val_f1 = f1_score(all_labels, all_preds, average='macro')
    
    history['train_loss'].append(train_loss_avg)
    history['val_loss'].append(val_loss_avg)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)
    
    # Save best model
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, 'dga_best_model.pt'))
        print(f'  ★ New best model saved (F1: {best_f1:.4f})')
    
    print(f'Epoch {epoch+1:2d}/{EPOCHS} | '
          f'Train Loss: {train_loss_avg:.4f} Acc: {train_acc:.4f} | '
          f'Val Loss: {val_loss_avg:.4f} Acc: {val_acc:.4f} F1: {val_f1:.4f}')

total_time = time.time() - start_time
print(f'\n✅ Training complete in {total_time/60:.1f} min')
print(f'Best Macro F1: {best_f1:.4f}')

In [ ]:
# ============================================================
# Training curves (for video screenshot)
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].plot(history['train_loss'], label='Train', linewidth=2)
axes[0].plot(history['val_loss'], label='Validation', linewidth=2)
axes[0].set_title('Loss', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['train_acc'], label='Train', linewidth=2)
axes[1].plot(history['val_acc'], label='Validation', linewidth=2)
axes[1].set_title('Accuracy', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(history['val_f1'], label='Macro F1', linewidth=2, color='green')
axes[2].set_title('Validation F1 Score', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Epoch')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle('DGA Detector (CNN-BiLSTM) — Training Curves', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'dga_training_curves.png'), dpi=150)
plt.show()
print('📸 Saved dga_training_curves.png')

## 5. Evaluation

In [ ]:
# ============================================================
# Load best model and evaluate
# ============================================================

model.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, 'dga_best_model.pt')))
model.eval()

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x = batch_x.to(device)
        logits = model(batch_x)
        probs = torch.softmax(logits, dim=1)
        preds = logits.argmax(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch_y.numpy())
        all_probs.extend(probs.cpu().numpy())

CLASS_NAMES = ['Benign', 'DGA', 'DNS Tunnel']

print('═' * 60)
print('DGA DETECTOR — EVALUATION RESULTS')
print('═' * 60)
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

final_f1 = f1_score(all_labels, all_preds, average='macro')
print(f'Macro F1: {final_f1:.4f}')

In [ ]:
# ============================================================
# Confusion Matrix (for video screenshot)
# ============================================================

fig, ax = plt.subplots(figsize=(8, 6))
cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm, annot=True, fmt=',d', cmap='Oranges',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_title(f'DGA Detector — Confusion Matrix (Macro F1: {final_f1:.4f})',
             fontsize=14, fontweight='bold')
ax.set_ylabel('True Label')
ax.set_xlabel('Predicted Label')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'dga_confusion_matrix.png'), dpi=150)
plt.show()
print('📸 Saved dga_confusion_matrix.png')

In [ ]:
!pip install -q onnxscript


## 6. Export to ONNX

In [ ]:
# ============================================================
# Export to ONNX (legacy exporter — compatible with BiLSTM)
# ============================================================

model.eval()
model_cpu = model.cpu()

dummy_input = torch.randint(0, VOCAB_SIZE, (1, MAX_LEN), dtype=torch.long)
onnx_path = os.path.join(OUTPUT_DIR, 'dga_cnn_bilstm.onnx')

torch.onnx.export(
    model_cpu,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=['domain_chars'],
    output_names=['logits'],
    dynamic_axes={
        'domain_chars': {0: 'batch_size'},
        'logits': {0: 'batch_size'}
    },
    dynamo=False  # Force legacy TorchScript-based exporter
)

print(f'✅ ONNX saved: {onnx_path} ({os.path.getsize(onnx_path)/1024/1024:.1f} MB)')


In [ ]:
# ============================================================
# Verify ONNX + Benchmark inference speed
# ============================================================

import onnxruntime as ort
import timeit

sess = ort.InferenceSession(onnx_path)
input_name = sess.get_inputs()[0].name

# Test prediction
test_domains_check = ['google.com', 'xk4jf9a2m.biz', 'aGVsbG8.d29ybGQ.tunnel-cdn.com']
for d in test_domains_check:
    encoded = np.array([encode_domain(d)], dtype=np.int64)
    output = sess.run(None, {input_name: encoded})
    probs = np.exp(output[0][0]) / np.exp(output[0][0]).sum()  # softmax
    pred_class = CLASS_NAMES[np.argmax(probs)]
    print(f'{d:40s} → {pred_class} (confidence: {probs.max():.3f})')

# Benchmark
single = np.array([encode_domain('google.com')], dtype=np.int64)
n_runs = 1000
total_time = timeit.timeit(lambda: sess.run(None, {input_name: single}), number=n_runs)
avg_ms = (total_time / n_runs) * 1000
print(f'\n⚡ ONNX inference: {avg_ms:.3f} ms/domain ({1000/avg_ms:,.0f} domains/sec)')

In [ ]:
# ============================================================
# Save metrics
# ============================================================

dga_metrics = {
    'model_name': 'NetSentinel DGA Detector',
    'model_type': 'CNN-BiLSTM (Character-level)',
    'version': '1.0.0',
    'architecture': {
        'embedding_dim': 64,
        'cnn_filters': 128,
        'cnn_kernels': [3, 4, 5],
        'lstm_hidden': 64,
        'lstm_layers': 2,
        'bidirectional': True,
    },
    'classes': CLASS_NAMES,
    'macro_f1': float(final_f1),
    'best_val_f1': float(best_f1),
    'training_time_minutes': float(total_time / 60),
    'epochs': EPOCHS,
    'inference_ms': float(avg_ms),
    'max_domain_length': MAX_LEN,
    'vocab_size': VOCAB_SIZE,
}

with open(os.path.join(OUTPUT_DIR, 'dga_metrics.json'), 'w') as f:
    json.dump(dga_metrics, f, indent=2)

print('\n' + '╔' + '═'*58 + '╗')
print('║' + ' MODEL CARD: NetSentinel DGA Detector'.center(58) + '║')
print('╠' + '═'*58 + '╣')
print(f'║  Architecture:  CNN-BiLSTM (char-level)'.ljust(59) + '║')
print(f'║  Classes:       Benign / DGA / DNS Tunnel'.ljust(59) + '║')
print(f'║  Macro F1:      {final_f1:.4f}'.ljust(59) + '║')
print(f'║  Inference:     {avg_ms:.3f} ms/domain'.ljust(59) + '║')
print(f'║  Training:      {total_time/60:.1f} min on {device}'.ljust(59) + '║')
print(f'║  Export:        ONNX (opset 17)'.ljust(59) + '║')
print('╚' + '═'*58 + '╝')

print('\n📁 Output files:')
for f in os.listdir(OUTPUT_DIR):
    print(f'  {f} ({os.path.getsize(os.path.join(OUTPUT_DIR, f))/1024:.1f} KB)')

In [ ]:
import zipfile
from IPython.display import FileLink

zip_path = '/kaggle/working/netsentinel_dga_output.zip'

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for file in os.listdir(OUTPUT_DIR):
        filepath = os.path.join(OUTPUT_DIR, file)
        zf.write(filepath, file)
        print(f'  Added: {file} ({os.path.getsize(filepath)/1024:.1f} KB)')

print(f'\n✅ Zip created: {zip_path} ({os.path.getsize(zip_path)/1024/1024:.1f} MB)')
FileLink(zip_path)
